**Step 1: Prepare the Evaluation Set**

This notebook turns the Gemini transcription into the set of people the name-search experiments search for. It reads `../../ocr_pipeline/extracted_text.csv`, which it never modifies, normalises every name with the rules the production import will use, merges rows that describe the same person on two stones, and splits the people into a training, a validation and a test set. Training people only teach the archive-learned dictionary (step 3); validation people are searched for while the search is tuned and chosen (steps 5–7); test people are searched for once, at the end (step 8). It outputs:

1. `./outputs/step_1_persons.csv`: one row per person, with section, graves, extraction confidence, split and normalised name tokens
2. `./outputs/step_1_profile.csv`: counts describing the transcription and the split

**1. Configuration and Helpers**

This cell defines the settings and the normalisation that the later cells use.

1. `normalise_latin` and `normalise_hebrew`: Latin text loses its accents, case and punctuation; Hebrew text loses its points and punctuation, its final letters become medial forms and the Yiddish ligatures become letter pairs
2. `display_form`: the spelling an embedding model reads, which keeps accents and Hebrew final letters and turns a name carved in capitals (`REBECA`) into title case (`Rebeca`)
3. `name_tokens`: splits a name field into tokens, dropping single-letter initials and, in surnames, the married-name particle (`de`, `דע`), so that "S. de Blejer" becomes [blejer]

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  paths, settings and name normalisation
#
# Every name token keeps two spellings: a normalised one, which exact and
# phonetic matching compare, and a display one, which embedding models
# read. Script is detected per token, so a Latin word in a Hebrew column is
# still treated as Latin.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
SOURCE_CSV     = Path("../../ocr_pipeline/extracted_text.csv")   # read only
OUTPUTS_DIR    = Path("outputs")                                 # every step's outputs
SEED           = 333     # seed of the split
MIN_CONFIDENCE = 85      # eligibility: extraction confidence, 0–100
RULE           = "─" * 72

NAME_COLUMNS = {"given_name_es": "given", "given_name_he": "given",
                "surname_es": "surname", "surname_he": "surname"}
SURNAME_PARTICLES = {"de", "דע"}                       # dropped from surnames
FINAL_TO_MEDIAL   = str.maketrans("ךםןףץ", "כמנפצ")     # final letters → medial forms
MAQAF             = "\N{HEBREW PUNCTUATION MAQAF}"     # the Hebrew hyphen
HEBREW_LETTERS    = "\N{HEBREW LETTER ALEF}-\N{HEBREW LETTER TAV}"            # a regex range
HEBREW_MARKS      = re.compile("[\N{HEBREW ACCENT ETNAHTA}-\N{HEBREW POINT QAMATS QATAN}]")
HEBREW_NON_LETTER = re.compile(f"[^{HEBREW_LETTERS} ]")  # anything but a letter or space
TOKEN_SPLIT       = re.compile(rf"[\s\-{MAQAF}]+")       # whitespace, hyphen, maqaf
YIDDISH_LIGATURES = {"\N{HEBREW LIGATURE YIDDISH DOUBLE VAV}": "וו",
                     "\N{HEBREW LIGATURE YIDDISH VAV YOD}": "וי",
                     "\N{HEBREW LIGATURE YIDDISH DOUBLE YOD}": "יי"}


def token_script(text: str) -> str:
    """Return `hebrew` if `text` holds any Hebrew character, else `latin`."""
    return "hebrew" if any(0x0590 <= ord(char) <= 0x05FF for char in text) else "latin"


def normalise_latin(text: str) -> str:
    """Normalise Latin-script text: no accents, lower case, letters only."""
    decomposed = unicodedata.normalize("NFKD", text.replace("-", " "))
    letters = "".join(char for char in decomposed
                      if not unicodedata.combining(char) and (char.isalpha() or char.isspace()))
    return " ".join(letters.lower().split())


def normalise_hebrew(text: str) -> str:
    """Normalise Hebrew-script text: no points, medial letters, letters only.

    NFKD first splits presentation forms such as `וּ` (one code point) into
    the letter and its point, so that removing the point keeps the letter.
    """
    text = unicodedata.normalize("NFKD", text).replace(MAQAF, " ").replace("-", " ")
    text = HEBREW_MARKS.sub("", text)
    for ligature, letters in YIDDISH_LIGATURES.items():
        text = text.replace(ligature, letters)
    return " ".join(HEBREW_NON_LETTER.sub("", text.translate(FINAL_TO_MEDIAL)).split())


def display_form(piece: str, script: str) -> str:
    """Return the spelling of one recorded token that an embedding model reads."""
    piece = unicodedata.normalize("NFC", piece)
    if script == "hebrew":
        piece = HEBREW_MARKS.sub("", piece)
        for ligature, letters in YIDDISH_LIGATURES.items():
            piece = piece.replace(ligature, letters)
        return re.sub(f"[^{HEBREW_LETTERS}]", "", piece)
    piece = "".join(char for char in piece if char.isalpha() or char == "'")
    return piece.capitalize() if piece.isupper() else piece


def name_tokens(raw: str, column: str) -> list[dict]:
    """Split one name field into tokens.

    Args:
        raw: the field as transcribed, e.g. `S. de Blejer`.
        column: the transcription column, e.g. `surname_es`.

    Returns:
        One dictionary per token: field, column, script, norm and display.
    """
    field_name, tokens = NAME_COLUMNS[column], []
    for piece in TOKEN_SPLIT.split(raw.strip()):
        script = token_script(piece)
        norm = (normalise_hebrew(piece) if script == "hebrew"
                else normalise_latin(piece)).replace(" ", "")
        if len(norm) < 2 or (field_name == "surname" and norm in SURNAME_PARTICLES):
            continue
        tokens.append({"field": field_name, "column": column, "script": script, "norm": norm,
                       "display": display_form(piece, script)})
    return tokens


def row_tokens(row: dict) -> list[dict]:
    """Return every name token of one transcription row."""
    return [token for column in NAME_COLUMNS for token in name_tokens(row[column], column)]


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


with SOURCE_CSV.open(encoding="utf-8", newline="") as _file:
    source_rows = list(csv.DictReader(_file))
print(f"✓ Configuration and helpers loaded ({len(source_rows)} transcription rows).")

✓ Configuration and helpers loaded (825 transcription rows).


**2. Eligible Rows and Duplicate Persons**

This cell takes `source_rows` and keeps the rows that record a given name and a surname, in either script, with an extraction confidence of at least `MIN_CONFIDENCE`. Rows that describe the same person on two stones are merged into one person: two rows merge when their given name, last surname token and year of death agree in the same script. It produces:

1. `persons`: one dictionary per person, numbered `P0001`, `P0002`, ... in grave order, with the union of its rows' tokens
2. each person's script coverage (`latin`, `hebrew` or `both`) and family key (its last surname token, the husband's name in "X de Y"), used by the split

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Eligible rows and duplicate persons  —  rows merged on name and death year
#
# The Serebrinsky brothers, named on both stones 7 and 8 of the children's
# section, are the example: each becomes one person with two graves.
# ─────────────────────────────────────────────────────────────────────────────
if "source_rows" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")


def is_eligible(row: dict, tokens: list[dict]) -> bool:
    """Return True if a row records a given name and a surname with enough confidence."""
    fields = {token["field"] for token in tokens}
    return fields == {"given", "surname"} and int(row["confidence"] or 0) >= MIN_CONFIDENCE


def merge_key(row: dict, tokens: list[dict], script: str) -> tuple | None:
    """Return the duplicate key of a row in one script, or None if incomplete."""
    given = [t["norm"] for t in tokens if t["field"] == "given" and t["script"] == script]
    surname = [t["norm"] for t in tokens if t["field"] == "surname" and t["script"] == script]
    year = row["death_year"].strip()
    return (script, given[0], surname[-1], year) if given and surname and year else None


def merge_rows(indices: list[int]) -> list[list[int]]:
    """Group row indices that share a duplicate key, transitively (a union–find).

    Returns:
        The groups of row indices, in order of their first row.
    """
    parent = {index: index for index in indices}

    def root(index: int) -> int:
        while parent[index] != index:
            index = parent[index]
        return index

    first_with_key = {}
    for index in indices:
        for script in ("latin", "hebrew"):
            key = merge_key(source_rows[index], source_tokens[index], script)
            if key in first_with_key:
                parent[root(index)] = root(first_with_key[key])
            elif key:
                first_with_key[key] = index
    groups = defaultdict(list)
    for index in indices:
        groups[root(index)].append(index)
    return sorted(groups.values(), key=lambda group: group[0])


def make_person(number: int, rows: list[int]) -> dict:
    """Return one person built from its merged rows."""
    tokens, seen = [], set()
    for index in rows:
        for token in source_tokens[index]:
            if (token["column"], token["norm"]) not in seen:
                seen.add((token["column"], token["norm"]))
                tokens.append(token)
    scripts = {token["script"] for token in tokens}
    surnames = [t["norm"] for t in tokens if t["field"] == "surname"]
    latin_surnames = [t["norm"] for t in tokens
                      if t["field"] == "surname" and t["script"] == "latin"]
    return {"person_id": f"P{number:04d}", "section": source_rows[rows[0]]["section"],
            "graves": json.dumps(sorted({"/".join(source_rows[i][k] for k in ("section", "row",
                                                                             "stone"))
                                         for i in rows})),
            "rows": json.dumps(rows),
            "confidence": min(int(source_rows[i]["confidence"]) for i in rows),
            "coverage": "both" if len(scripts) == 2 else scripts.pop(),
            "family_key": (latin_surnames or surnames)[-1], "split": "",
            "tokens": json.dumps(tokens, ensure_ascii=False)}


source_tokens = [row_tokens(row) for row in source_rows]
eligible = [i for i, row in enumerate(source_rows) if is_eligible(row, source_tokens[i])]
persons = [make_person(n, rows) for n, rows in enumerate(merge_rows(eligible), start=1)]

print(f"{RULE}\nEligible rows and persons\n{RULE}")
print(f"  rows in the transcription       {len(source_rows):>5}")
print(f"  eligible rows                   {len(eligible):>5}")
print(f"  persons after merging           {len(persons):>5}")
print(f"  persons with two or more rows   "
      f"{sum(len(json.loads(p['rows'])) > 1 for p in persons):>5}")
print(f"{RULE}\n  {len(persons)} persons from {len(eligible)} eligible rows")

────────────────────────────────────────────────────────────────────────
Eligible rows and persons
────────────────────────────────────────────────────────────────────────
  rows in the transcription         825
  eligible rows                     639
  persons after merging             637
  persons with two or more rows       2
────────────────────────────────────────────────────────────────────────
  637 persons from 639 eligible rows


**3. Train, Validation and Test Split**

This cell takes `persons` and splits them into five groups of about equal size with stratified group k-fold (seed `SEED`): the strata are section × script coverage, so every group mixes sections and scripts like the whole archive, and the groups are family keys, so that one family surname never spans two splits. One group becomes the test set, one the validation set and three the training set (about 60/20/20). It writes:

1. `./outputs/step_1_persons.csv`: every person with its split
2. `./outputs/step_1_profile.csv`: the counts of the transcription, the persons and the split

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Train, validation and test split  —  60/20/20, family surnames kept whole
# ─────────────────────────────────────────────────────────────────────────────
if "persons" not in globals():
    raise RuntimeError("Run the 'Eligible Rows and Duplicate Persons' cell first.")

SPLIT_OF_GROUP = {0: "test", 1: "validation", 2: "train", 3: "train", 4: "train"}

_strata = [f"{p['section']}|{p['coverage']}" for p in persons]
_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for _group, (_, _members) in enumerate(_splitter.split(_strata, _strata,
                                                       [p["family_key"] for p in persons])):
    for _index in _members:
        persons[_index]["split"] = SPLIT_OF_GROUP[_group]
write_csv(OUTPUTS_DIR / "step_1_persons.csv", persons)

split_sizes = Counter(p["split"] for p in persons)
profile = [{"metric": "transcription rows", "value": len(source_rows)},
           {"metric": "graves", "value": len({(r["section"], r["row"], r["stone"])
                                             for r in source_rows})},
           {"metric": "eligible rows", "value": len(eligible)},
           {"metric": "persons", "value": len(persons)},
           *({"metric": f"persons with {c} names", "value": n}
             for c, n in sorted(Counter(p["coverage"] for p in persons).items())),
           *({"metric": f"{s} persons", "value": split_sizes[s]}
             for s in ("train", "validation", "test"))]
write_csv(OUTPUTS_DIR / "step_1_profile.csv", profile)

print(f"{RULE}\nSplit: section × coverage strata, family keys kept whole\n{RULE}")
for _split in ("train", "validation", "test"):
    _members = [p for p in persons if p["split"] == _split]
    print(f"  {_split:<11} {len(_members):>4} persons   "
          + "  ".join(f"{c} {n}"
                      for c, n in sorted(Counter(p['coverage'] for p in _members).items())))
print(f"{RULE}\n  written to {OUTPUTS_DIR}/step_1_persons.csv and step_1_profile.csv")

────────────────────────────────────────────────────────────────────────
Split: section × coverage strata, family keys kept whole
────────────────────────────────────────────────────────────────────────
  train        381 persons   both 279  hebrew 38  latin 64
  validation   126 persons   both 93  hebrew 12  latin 21
  test         130 persons   both 95  hebrew 13  latin 22
────────────────────────────────────────────────────────────────────────
  written to outputs/step_1_persons.csv and step_1_profile.csv


**4. Consistency Check**

This cell reads `./outputs/step_1_persons.csv` back and checks that every eligible row belongs to exactly one person, that every person has a split, and that no family key spans two splits.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the persons file against the eligible rows
# ─────────────────────────────────────────────────────────────────────────────
if "split_sizes" not in globals():
    raise RuntimeError("Run every cell above first.")

with (OUTPUTS_DIR / "step_1_persons.csv").open(encoding="utf-8", newline="") as _file:
    _stored = list(csv.DictReader(_file))
_row_owners = Counter(i for p in _stored for i in json.loads(p["rows"]))
_splits_of_family = defaultdict(set)
for _person in _stored:
    _splits_of_family[_person["family_key"]].add(_person["split"])
_problems = ([] if sorted(_row_owners) == eligible and set(_row_owners.values()) == {1}
             else ["eligible rows and persons disagree"])
_problems += [f"{p['person_id']} has no split" for p in _stored if not p["split"]]
_problems += [f"family {k} spans {sorted(v)}" for k, v in _splits_of_family.items() if len(v) > 1]
print(f"{RULE}\nConsistency: {OUTPUTS_DIR}/step_1_persons.csv\n{RULE}")
for _problem in _problems:
    print(f"  !! {_problem}")
print(f"  {'OK' if not _problems else '!!'} {len(_stored)} persons, "
      f"{len(_row_owners)} rows, {len(_splits_of_family)} family keys")
print(f"\n✓ Evaluation set complete ({split_sizes['train']} train, "
      f"{split_sizes['validation']} validation, {split_sizes['test']} test persons).")
print(f"  consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: outputs/step_1_persons.csv
────────────────────────────────────────────────────────────────────────
  OK 637 persons, 639 rows, 468 family keys

✓ Evaluation set complete (381 train, 126 validation, 130 test persons).
  consistent: True
